In [1]:
# automatically reloads all modules before executing a new cell
%load_ext autoreload
%autoreload 2

In [1]:
import datasets
from datasets import load_dataset
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
from collections import defaultdict
import torch
import torch.nn.functional as F
import random
from torch.utils.data import DataLoader
from tqdm import tqdm

In [2]:
def process_photometry(examples, seq_len=200, how='center'):
    """
    Processes photometry data by trimming or padding sequences to a fixed length.

    Parameters:
    - examples (dict): A dictionary containing a "photometry" key.
    - seq_len (int): The target sequence length (default: 200).
    - how (str): The trimming strategy:
        - 'center' (default): Trims the sequence symmetrically around the center.
        - 'random': Selects a random `seq_len`-length segment if the sequence is too long.

    Returns:
    - dict: Updated example with:
        - "photometry": Trimmed/padded photometry tensor (shape: [seq_len]).
        - "photometry_mask": Mask tensor (1s for real values, 0s for padded values).
    """
    photometry_all = []
    mask_all = []
    
    for photometry in examples['photometry']:
        if type(photometry) == list:
            photometry = torch.tensor(photometry, dtype=torch.float32)
        mask = torch.ones(seq_len, dtype=torch.float32)
    
        if photometry.shape[0] > seq_len:
            # Determine trimming start index
            if how == 'center':
                start_idx = (photometry.shape[0] - seq_len) // 2
            else:  # 'random'
                start_idx = random.randint(0, photometry.shape[0] - seq_len)
            
            # Trim sequence
            photometry = photometry[start_idx : start_idx + seq_len]
        else:
            # Update mask for padded values
            mask[photometry.shape[0]:] = 0
            
            # Pad sequence with zeros
            padding_length = seq_len - photometry.shape[0]
            photometry = F.pad(photometry, (0, 0, 0, padding_length), value=0)
        
        photometry_all.append(photometry)
        mask_all.append(mask)
        
    # Update example dictionary
    examples['photometry'] = photometry_all
    examples['photometry_mask'] = mask_all
    
    return examples

In [29]:
def process_train(examples, seq_len=200, how='random'):
    examples = process_photometry(examples, seq_len=seq_len, how=how)
    examples['spectra'] = [torch.tensor(el) for el in examples['spectra']]
    examples['metadata'] = [torch.tensor(el) for el in examples['metadata']]
    examples['label'] = [torch.tensor(el) for el in examples['label']]
    
    return examples

In [39]:
train = load_dataset('MeriDK/AstroM3Processed', name='full_42', split='train')
val = load_dataset('MeriDK/AstroM3Processed', name='full_42', split='validation')
test = load_dataset('MeriDK/AstroM3Processed', name='full_42', split='test')

In [40]:
val = val.with_format('torch')
test = test.with_format('torch')

val = val.map(process_photometry, batched=True, fn_kwargs={'seq_len': 200, 'how': 'center'})
test = test.map(process_photometry, batched=True, fn_kwargs={'seq_len': 200, 'how': 'center'})

In [41]:
train.set_transform(lambda examples: process_train(examples, seq_len=200, how='random'))

In [42]:
photometry = train[0]['photometry']
photometry_mask = train[0]['photometry_mask']
spectra = train[0]['spectra']
metadata = train[0]['metadata']
label = train[0]['label']

In [48]:
photometry = val[0]['photometry']
photometry_mask = val[0]['photometry_mask']
spectra = val[0]['spectra']
metadata = val[0]['metadata']
label = val[0]['label']

In [49]:
photometry.shape, photometry.dtype

(torch.Size([200, 9]), torch.float32)

In [50]:
photometry_mask.shape, photometry_mask.dtype

(torch.Size([200]), torch.float32)

In [51]:
spectra.shape, spectra.dtype

(torch.Size([3, 2575]), torch.float32)

In [52]:
metadata.shape, metadata.dtype

(torch.Size([34]), torch.float32)

In [53]:
label, label.dtype

(tensor(3), torch.int64)

In [5]:
for i in range(10):
    print(val[i]['photometry'].shape)

torch.Size([200, 9])
torch.Size([200, 9])
torch.Size([200, 9])
torch.Size([200, 9])
torch.Size([200, 9])
torch.Size([200, 9])
torch.Size([200, 9])
torch.Size([200, 9])
torch.Size([200, 9])
torch.Size([200, 9])


In [37]:
dl = DataLoader(train, batch_size=512, num_workers=4, shuffle=False)

In [38]:
for batch in tqdm(dl):
    pass

100%|███████████████████████████████████████████████████████████████████████████████████| 34/34 [00:16<00:00,  2.09it/s]


In [54]:
batch = next(iter(dl))

In [56]:
batch['photometry'].shape

torch.Size([512, 200, 9])

In [176]:
train.set_transform(process_photometry)

In [65]:
train.features["label"].names

['DSCT', 'EA', 'EB', 'EW', 'HADS', 'M', 'ROT', 'RRAB', 'RRC', 'SR']

In [109]:
def preprocess_spectra(example):
    """
    Preprocess spectral data.

    Steps:
    - Interpolate flux and flux error to a fixed wavelength grid (3850 to 9000 Å).
    - Normalize flux using mean and median absolute deviation (MAD).
    - Append MAD as an auxiliary feature.

    Args:
        spectra (numpy.ndarray): Original spectra data of shape (N, 3) where columns are (wavelength, flux, flux_error).

    Returns:
        numpy.ndarray: Preprocessed spectral data of shape (3, num_wavelengths).
    """
    spectra = example['spectra']
    wavelengths = spectra[:, 0]
    flux = spectra[:, 1]
    flux_err = spectra[:, 2]

    # Interpolate flux and flux error onto a fixed grid
    new_wavelengths = np.arange(3850, 9000, 2)
    flux = np.interp(new_wavelengths, wavelengths, flux)
    flux_err = np.interp(new_wavelengths, wavelengths, flux_err)

    # Normalize flux and flux error
    mean = np.mean(flux)
    mad = stats.median_abs_deviation(flux[flux != 0])

    flux = (flux - mean) / mad
    flux_err = flux_err / mad
    aux_values = np.full_like(flux, np.log10(mad))  # Store MAD as an auxiliary feature

    # Stack processed data into a single array
    spectra = np.vstack([flux, flux_err, aux_values])
    example['spectra'] = spectra
    
    return example

In [108]:
def preprocess_lc(example):
    """
    Preprocess photometry (light curve) data.

    Steps:
    - Remove duplicate time entries.
    - Sort by Heliocentric Julian Date (HJD).
    - Normalize flux and flux error using mean and median absolute deviation (MAD).
    - Scale time values between 0 and 1.
    - Append auxiliary features (log MAD and time span delta_t).

    Args:
        X (numpy.ndarray): Original photometry data of shape (N, 3) where columns are (HJD, flux, flux_error).
        aux_values (numpy.ndarray): Auxiliary metadata features.

    Returns:
        X (numpy.ndarray): Preprocessed photometry data.
    """
    X = example['photometry']
    aux_values = np.stack(list(example['metadata']['photo_cols'].values()))
    
    # Remove duplicate entries
    X = np.unique(X, axis=0)

    # Sort based on HJD
    sorted_indices = np.argsort(X[:, 0])
    X = X[sorted_indices]

    # Normalize flux and flux error
    mean = X[:, 1].mean()
    mad = stats.median_abs_deviation(X[:, 1])
    X[:, 1] = (X[:, 1] - mean) / mad
    X[:, 2] = X[:, 2] / mad

    # Compute delta_t (time span of the light curve in years)
    delta_t = (X[:, 0].max() - X[:, 0].min()) / 365

    # Scale time from 0 to 1
    X[:, 0] = (X[:, 0] - X[:, 0].min()) / (X[:, 0].max() - X[:, 0].min())

    # Add MAD and delta_t to auxiliary metadata features
    aux_values = np.concatenate((aux_values, [np.log10(mad), delta_t]))

    # Add auxiliary features to the sequence
    aux_values = np.tile(aux_values, (X.shape[0], 1))
    X = np.concatenate((X, aux_values), axis=-1)
    
    example['photometry'] = X
    return example

In [95]:
METADATA_FUNC = {
    "abs": [
        "mean_vmag",
        "phot_g_mean_mag",
        "phot_bp_mean_mag",
        "phot_rp_mean_mag",
        "j_mag",
        "h_mag",
        "k_mag",
        "w1_mag",
        "w2_mag",
        "w3_mag",
        "w4_mag",
    ],
    "cos": ["l"],
    "sin": ["b"],
    "log": ["period"]
}

In [94]:
def transform_metadata(example):
    """
    Transforms the metadata of an example based on METADATA_FUNC.
    """
    metadata = example["metadata"]

    # Process 'abs' transformation on meta_cols:
    # Note: This transformation uses 'parallax' from meta_cols.
    for col in METADATA_FUNC["abs"]:
        if col in metadata["meta_cols"]:
            # Use np.where to avoid issues when parallax is non-positive.
            metadata["meta_cols"][col] = (
                metadata["meta_cols"][col]
                - 10
                + 5 * np.log10(np.where(metadata["meta_cols"]["parallax"] <= 0, 1, metadata["meta_cols"]["parallax"]))
            )

    # Process 'cos' transformation on meta_cols:
    for col in METADATA_FUNC["cos"]:
        if col in metadata["meta_cols"]:
            metadata["meta_cols"][col] = np.cos(np.radians(metadata["meta_cols"][col]))

    # Process 'sin' transformation on meta_cols:
    for col in METADATA_FUNC["sin"]:
        if col in metadata["meta_cols"]:
            metadata["meta_cols"][col] = np.sin(np.radians(metadata["meta_cols"][col]))

    # Process 'log' transformation on photo_cols:
    for col in METADATA_FUNC["log"]:
        if col in metadata["photo_cols"]:
            metadata["photo_cols"][col] = np.log10(metadata["photo_cols"][col])

    # Update the example with the transformed metadata.
    example["metadata"] = metadata
    return example

In [101]:
def compute_metadata_stats(ds):
    """
    Compute the mean and standard deviation for each column in meta_cols and photo_cols.
    """
    meta_vals = defaultdict(list)
    photo_vals = defaultdict(list)
    
    # Accumulate values for each column
    for example in ds:
        meta = example["metadata"]["meta_cols"]
        photo = example["metadata"]["photo_cols"]
        for col, value in meta.items():
            meta_vals[col].append(value)
        for col, value in photo.items():
            photo_vals[col].append(value)
    
    # Compute mean and standard deviation for each column
    stats = {"meta_cols": {}, "photo_cols": {}}
    for col, values in meta_vals.items():
        arr = np.stack(values)
        stats["meta_cols"][col] = {"mean": arr.mean(), "std": arr.std()}
    for col, values in photo_vals.items():
        arr = np.stack(values)
        stats["photo_cols"][col] = {"mean": arr.mean(), "std": arr.std()}
    
    return stats

In [105]:
def normalize_metadata(example, stats):
    """
    Normalize metadata values using z-score normalization:
    (value - mean) / std.
    
    The 'stats' parameter should be a dictionary with computed means and stds for both meta_cols and photo_cols.
    """
    metadata = example["metadata"]
    
    # Normalize meta_cols
    for col, value in metadata["meta_cols"].items():
        mean = stats["meta_cols"][col]["mean"]
        std = stats["meta_cols"][col]["std"]
        metadata["meta_cols"][col] = (metadata["meta_cols"][col] - mean) / std

    # Normalize photo_cols
    for col, value in metadata["photo_cols"].items():
        mean = stats["photo_cols"][col]["mean"]
        std = stats["photo_cols"][col]["std"]
        metadata["photo_cols"][col] = (metadata["photo_cols"][col] - mean) / std
    
    example["metadata"] = metadata
    return example

In [110]:
def preprocess_metadata(example):
    """
    Extract the values from 'meta_cols' and stack them into a numpy array. 
    """
    example['metadata'] = np.stack(list(example['metadata']['meta_cols'].values()))
    return example

In [88]:
ds = load_dataset('MeriDK/AstroM3Dataset', name='sub10_42', trust_remote_code=True, num_proc=16)

In [89]:
ds = ds.with_format('numpy')

In [92]:
ds = ds.map(transform_metadata, num_proc=16)

Map (num_proc=16):   0%|          | 0/1660 [00:00<?, ? examples/s]

Map (num_proc=16):   0%|          | 0/210 [00:00<?, ? examples/s]

Map (num_proc=16):   0%|          | 0/220 [00:00<?, ? examples/s]

In [106]:
stats = compute_metadata_stats(ds['train'])
ds = ds.map(lambda example: normalize_metadata(example, stats))

Map:   0%|          | 0/1660 [00:00<?, ? examples/s]

Map:   0%|          | 0/210 [00:00<?, ? examples/s]

Map:   0%|          | 0/220 [00:00<?, ? examples/s]

In [34]:
ds = ds.map(preprocess_spectra, num_proc=16)

Map (num_proc=16):   0%|          | 0/1660 [00:00<?, ? examples/s]

Map (num_proc=16):   0%|          | 0/210 [00:00<?, ? examples/s]

Map (num_proc=16):   0%|          | 0/220 [00:00<?, ? examples/s]

In [37]:
ds = ds.cast_column('spectra', datasets.Array2D(shape=(3, 2575), dtype='float32'))

Casting the dataset:   0%|          | 0/1660 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/210 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/220 [00:00<?, ? examples/s]

In [48]:
ds = ds.map(preprocess_lc, num_proc=16)

Map (num_proc=16):   0%|          | 0/1660 [00:00<?, ? examples/s]

Map (num_proc=16):   0%|          | 0/210 [00:00<?, ? examples/s]

Map (num_proc=16):   0%|          | 0/220 [00:00<?, ? examples/s]

In [50]:
ds = ds.cast_column('photometry', datasets.Array2D(shape=(None, 9), dtype='float32'))

Casting the dataset:   0%|          | 0/1660 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/210 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/220 [00:00<?, ? examples/s]

In [53]:
ds = ds.map(preprocess_metadata, num_proc=16)

Map (num_proc=16):   0%|          | 0/1660 [00:00<?, ? examples/s]

Map (num_proc=16):   0%|          | 0/210 [00:00<?, ? examples/s]

Map (num_proc=16):   0%|          | 0/220 [00:00<?, ? examples/s]

In [56]:
ds = ds.cast_column('metadata', datasets.Sequence(feature=datasets.Value('float32'), length=34))

Casting the dataset:   0%|          | 0/1660 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/210 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/220 [00:00<?, ? examples/s]

In [59]:
ds = ds.cast_column('label', datasets.ClassLabel(names=['DSCT', 'EA', 'EB', 'EW', 'HADS', 'M', 'ROT', 'RRAB', 'RRC', 'SR']))

Casting the dataset:   0%|          | 0/1660 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/210 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/220 [00:00<?, ? examples/s]

In [61]:
ds.push_to_hub('MeriDK/AstroM3Processed', config_name='sub10_42_preprocessed')

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

CommitInfo(commit_url='https://huggingface.co/datasets/MeriDK/AstroM3Processed/commit/afbdbf81790531d845793c85fd26e2a2acdb2079', commit_message='Upload dataset', commit_description='', oid='afbdbf81790531d845793c85fd26e2a2acdb2079', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/MeriDK/AstroM3Processed', endpoint='https://huggingface.co', repo_type='dataset', repo_id='MeriDK/AstroM3Processed'), pr_revision=None, pr_num=None)

In [67]:
ds = load_dataset('MeriDK/AstroM3Processed', name='sub10_42_preprocessed')

README.md:   0%|          | 0.00/102k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1660 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/210 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/220 [00:00<?, ? examples/s]

In [68]:
ds['train'].features

{'photometry': Array2D(shape=(None, 9), dtype='float32', id=None),
 'spectra': Array2D(shape=(3, 2575), dtype='float32', id=None),
 'metadata': Sequence(feature=Value(dtype='float32', id=None), length=34, id=None),
 'label': ClassLabel(names=['DSCT', 'EA', 'EB', 'EW', 'HADS', 'M', 'ROT', 'RRAB', 'RRC', 'SR'], id=None)}

In [71]:
ds = ds.with_format('numpy')

In [72]:
ds['train'][0]

{'photometry': array([[ 0.0000000e+00,  5.2961022e-01,  1.3599986e+00, ...,
          9.4999999e-01, -7.5696146e-01,  5.7815070e+00],
        [ 1.8481223e-02,  1.1410401e+00,  1.3714271e+00, ...,
          9.4999999e-01, -7.5696146e-01,  5.7815070e+00],
        [ 6.7646012e-02, -2.0401793e+01,  1.0628560e+00, ...,
          9.4999999e-01, -7.5696146e-01,  5.7815070e+00],
        ...,
        [ 9.3531573e-01,  1.6381818e+00,  1.3771414e+00, ...,
          9.4999999e-01, -7.5696146e-01,  5.7815070e+00],
        [ 9.3768513e-01,  3.2096119e+00,  1.4057128e+00, ...,
          9.4999999e-01, -7.5696146e-01,  5.7815070e+00],
        [ 1.0000000e+00,  2.5238936e+00,  1.3942842e+00, ...,
          9.4999999e-01, -7.5696146e-01,  5.7815070e+00]], dtype=float32),
 'spectra': array([[ 1.4087214e+00,  1.4977813e+00,  1.5336988e+00, ...,
         -1.8422334e+00, -1.7332082e+00, -1.6669453e+00],
        [ 1.4011429e-06,  1.4323066e-06,  1.4426632e-06, ...,
          1.5738161e-06,  1.5311875e-06,  1

In [12]:
dataset['train'].info.features['photometry']

Array2D(shape=(None, 3), dtype='float32', id=None)

In [13]:
dataset.push_to_hub('MeriDK/AstroM3Processed', config_name='sub10_42')

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/datasets/MeriDK/AstroM3Processed/commit/d2b9eb68d33bfdef4499983f6724e33b5c0b6106', commit_message='Upload dataset', commit_description='', oid='d2b9eb68d33bfdef4499983f6724e33b5c0b6106', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/MeriDK/AstroM3Processed', endpoint='https://huggingface.co', repo_type='dataset', repo_id='MeriDK/AstroM3Processed'), pr_revision=None, pr_num=None)

In [19]:
processed_dataset = load_dataset('MeriDK/AstroM3Processed', name='sub10_66')

train-00000-of-00001.parquet:   0%|          | 0.00/84.3M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/12.4M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/11.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1670 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/200 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/220 [00:00<?, ? examples/s]

In [20]:
processed_dataset

DatasetDict({
    train: Dataset({
        features: ['photometry', 'spectra', 'metadata', 'label'],
        num_rows: 1670
    })
    validation: Dataset({
        features: ['photometry', 'spectra', 'metadata', 'label'],
        num_rows: 200
    })
    test: Dataset({
        features: ['photometry', 'spectra', 'metadata', 'label'],
        num_rows: 220
    })
})

In [7]:
dataset = load_dataset('/home/mariia/AstroM3Dataset/AstroM3Dataset.py', name='full_42', trust_remote_code=True)

ImportError: To be able to use AstroM3Dataset, you need to install the following dependency: utils.
Please install it using 'pip install utils' for instance.